# Chapter 8 &mdash; Anatomy of the RE-to-NFA Converter: a Mini-Compiler

**Concept 5 of the Chapter 8 decomposition:** *Anatomy of the RE-to-NFA Converter: a Mini-Compiler*

`re2nfa` is a lexer plus a parser whose production rules assemble NFA fragments.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-RE-Converter-Anatomy/Concept-RE-Converter-Anatomy.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.AnimateNFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateNFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateNFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


`re2nfa` is a small but complete **compiler**:

* a **lexer** (`lex()`) turning the RE text into tokens &mdash; `t_PLUS`, `t_STAR`,
  `t_LPAREN`, `t_RPAREN`, `t_EPS`, `t_STR`;
* a **parser** (`yacc()`) with one production per RE operator, whose **semantic
  action** is a fragment constructor from Concept 2.

The grammar layers encode precedence: `expression` handles `+`, `catexp` handles
juxtaposition, `ordyexp` handles `*` and parentheses. Union is loosest because it sits
at the top.

Reading this converter teaches you how *any* small language is implemented &mdash; and
Chapter 11's grammars are the same idea from the other side.

## 2. Definitions

### The tokens the lexer recognises

In [ ]:
import jove.Def_RE2NFA as C
for t in ['t_PLUS', 't_STAR', 't_LPAREN', 't_RPAREN', 't_EPS', 't_STR']:
    print("%-10s = %r" % (t, getattr(C, t)))
print("\ntoken list :", C.tokens)

### The grammar productions, read off the parser functions

In [ ]:
import inspect
def productions():
    for name in sorted(n for n in dir(C) if n.startswith('p_') and n != 'p_error'):
        doc = (getattr(C, name).__doc__ or '').strip()
        print("%-26s %s" % (name, doc))

<!-- nav-strip -->

---

&larr;&nbsp;[Ch8&nbsp;4.&nbsp;Regular Expressions Are Error-Prone, and That Is a Security Problem](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-RE-Are-Error-Prone/Concept-RE-Are-Error-Prone.ipynb) &nbsp;&middot;&nbsp; [**Chapter 8** index](https://github.com/ganeshutah/Jove/blob/master/Chapter8/README.md) &nbsp;&middot;&nbsp; [Ch8&nbsp;6.&nbsp;Error-Correcting Design I: the RE for "within Hamming Distance 2"](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-Hamming-Distance-RE/Concept-Hamming-Distance-RE.ipynb)&nbsp;&rarr;

---

## 3. Tests

The grammar, one production per line.

In [ ]:
productions()

The three layers **are** the precedence: `+` at the top, `*` at the bottom.

In [ ]:
print("expression : expression PLUS catexp     <- union, loosest")
print("catexp     : catexp ordyexp             <- concatenation")
print("ordyexp    : ordyexp STAR | ( expr )    <- star and grouping, tightest")
print()
print("so 0+1*  parses as 0 + (1*) :",
      sorted({s for s in ['', '0', '1', '11', '01'] if accepts_nfa(re2nfa('0+1*'), s)}))

Each production's **action** builds a fragment &mdash; the compiler's code generator.

In [ ]:
src = inspect.getsource(C.p_expression_plus)
print(src.strip()[:400])

The whole pipeline, end to end, on one expression.

In [ ]:
r = "(0+1)*1"
N = re2nfa(r)
print("RE     :", r)
print("tokens : lexed by lex(), parsed by yacc()")
print("NFA    : |Q| = %d, Q0 = %s, F = %s" % (len(N["Q"]), sorted(N["Q0"]), sorted(N["F"])))
assert accepts_nfa(N, '1') and accepts_nfa(N, '0001') and not accepts_nfa(N, '10')

A syntax error is reported by `p_error`, not by a crash deep inside.

In [ ]:
try:
    re2nfa("(0+1")
    print("no error raised")
except Exception as e:
    print("parse failure :", type(e).__name__, str(e)[:60])

## 4. Animation

The NFA the mini-compiler emitted.

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(re2nfa('(0+1)*1'), FuseEdges=True)

## 5. Exercises


1. Add a `?` (optional) operator. Which production and which constructor?
2. Why is `catexp` a separate layer rather than a precedence declaration?
3. Compare this converter with the `md2mc` parser. What do they share?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 246 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter8/Concept-RE-Converter-Anatomy')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')